#04 - Gold Layer - Store Dimension

Create a business-ready Store Dimension from the validated Silver store data.

**Source:** end-to-end_pipeline.silver.stores

**Target:** end-to-end_pipeline.gold.dim_store

**Model Role:** Dimension Table

**Business Key:** store_id

**Approach:** Profile → Inspect → Transform → Validate


**Purpose:**
Provide business-ready store attributes for analyzing sales performance by store, city, region, country, store type, and manager.

#Cell 1 - Profile Silver Store Data

**Description:**
Confirm that the Silver Stores table is ready to become a Gold dimension by checking key uniqueness, row count, and availability of the descriptive attributes required for reporting.

In [0]:
%sql

-- ============================================================
-- CELL 1: PROFILE SILVER STORES FOR GOLD MODELING
-- Purpose: Confirm dimension grain, key uniqueness,
--          and availability of business attributes
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT store_id) AS distinct_store_ids,
    COUNT(*) - COUNT(DISTINCT store_id) AS duplicate_store_ids,

    SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END)
        AS null_store_ids,

    COUNT(DISTINCT store_type)
        AS store_types,

    COUNT(DISTINCT region)
        AS regions,

    COUNT(DISTINCT country)
        AS countries,

    MIN(opening_date)
        AS earliest_opening_date,

    MAX(opening_date)
        AS latest_opening_date

FROM `end-to-end_pipeline`.silver.stores;

#Cell 2 - Inspect Store Business Attributes

**Description:**
Review how stores are distributed across store type, region, and country. This helps confirm that the dimension supports useful business slicing for the dashboard and Genie.

In [0]:
%sql

-- ============================================================
-- CELL 2: INSPECT STORE BUSINESS ATTRIBUTES
-- Purpose: Review store distribution across business groups
-- ============================================================

SELECT
    store_type,
    region,
    country,
    COUNT(*) AS store_count

FROM `end-to-end_pipeline`.silver.stores

GROUP BY
    store_type,
    region,
    country

ORDER BY
    country,
    region,
    store_type;

#Cell 3 - Transform Silver → Gold Store Dimension

**Description:**
Create the Gold Store Dimension at one row per store. No Silver cleaning is repeated. The table exposes the descriptive store attributes needed for business analysis while preserving store_id as the key that will connect to gold.fact_sales.

In [0]:
%sql

-- ============================================================
-- CELL 3: CREATE GOLD STORE DIMENSION
-- Grain: One row per store
-- Business Key: store_id
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.gold.dim_store AS

SELECT
    store_id,
    store_name,
    store_type,
    city,
    region,
    country,
    opening_date,
    YEAR(opening_date) AS opening_year,
    manager_name

FROM `end-to-end_pipeline`.silver.stores;

#Cell 3a - Add Business Metadata

**Description:**
Add table and column comments to improve discoverability in Genie and Databricks dashboards. These descriptions help users understand the business meaning of each attribute.

In [0]:
%sql

-- ============================================================
-- CELL 3a: ADD BUSINESS METADATA TO STORE DIMENSION
-- Purpose: Improve discoverability for Genie and dashboards
-- ============================================================

COMMENT ON TABLE `end-to-end_pipeline`.gold.dim_store IS 
'Store dimension providing location, type, and operational attributes for sales analysis. One row per store.';

ALTER TABLE `end-to-end_pipeline`.gold.dim_store 
  ALTER COLUMN store_id COMMENT 'Unique store identifier (business key)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_store 
  ALTER COLUMN store_name COMMENT 'Display name of the store';

ALTER TABLE `end-to-end_pipeline`.gold.dim_store 
  ALTER COLUMN store_type COMMENT 'Store format classification (e.g., Flagship, Standard, Express)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_store 
  ALTER COLUMN city COMMENT 'City where the store is located';

ALTER TABLE `end-to-end_pipeline`.gold.dim_store 
  ALTER COLUMN region COMMENT 'Geographic region of the store';

ALTER TABLE `end-to-end_pipeline`.gold.dim_store 
  ALTER COLUMN country COMMENT 'Country where the store is located';

ALTER TABLE `end-to-end_pipeline`.gold.dim_store 
  ALTER COLUMN opening_date COMMENT 'Date the store opened for business';

ALTER TABLE `end-to-end_pipeline`.gold.dim_store 
  ALTER COLUMN opening_year COMMENT 'Year the store opened (derived from opening_date)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_store 
  ALTER COLUMN manager_name COMMENT 'Name of the store manager';

#Cell 4 - Validate Gold Store Dimension

**Description:**
Validate that the Gold Store Dimension preserves one row per store, contains the required descriptive attributes, and has the same row coverage as the validated Silver source.

In [0]:
%sql

-- ============================================================
-- CELL 4: VALIDATE GOLD STORE DIMENSION
-- Purpose: Confirm dimension grain, key integrity,
--          and Silver → Gold completeness
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT store_id)
            AS distinct_store_ids,

        COUNT(*) - COUNT(DISTINCT store_id)
            AS duplicate_store_ids,

        SUM(
            CASE
                WHEN store_id IS NULL THEN 1
                ELSE 0
            END
        ) AS null_store_ids,

        SUM(
            CASE
                WHEN store_name IS NULL THEN 1
                ELSE 0
            END
        ) AS null_store_names,

        SUM(
            CASE
                WHEN store_type IS NULL THEN 1
                ELSE 0
            END
        ) AS null_store_types,

        SUM(
            CASE
                WHEN region IS NULL THEN 1
                ELSE 0
            END
        ) AS null_regions,

        SUM(
            CASE
                WHEN country IS NULL THEN 1
                ELSE 0
            END
        ) AS null_countries,

        SUM(
            CASE
                WHEN opening_year IS NULL THEN 1
                ELSE 0
            END
        ) AS null_opening_years

    FROM `end-to-end_pipeline`.gold.dim_store
),

source_check AS (

    SELECT
        COUNT(*) AS silver_rows

    FROM `end-to-end_pipeline`.silver.stores
)

SELECT
    v.*,
    s.silver_rows,

    CASE
        WHEN v.total_rows = s.silver_rows
            AND v.distinct_store_ids = v.total_rows
            AND v.duplicate_store_ids = 0
            AND v.null_store_ids = 0
            AND v.null_store_names = 0
            AND v.null_store_types = 0
            AND v.null_regions = 0
            AND v.null_countries = 0
            AND v.null_opening_years = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation v
CROSS JOIN source_check s;